# 🕵️ Dia 30 — Reconhecimento DNS com dig, nslookup e Certificate Transparency

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 30: DNS reconnaissance with dig, nslookup, and certificate transparency* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-30.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sábado, 10 de outubro. Um certificado gratuito, emitido na tarde de 4/out, cobre quatro nomes: o `serraazul-pix`, o `www`, um `painel`... e o `serraazul-seguranca`. Rafa: *"Isso é aresta com rótulo. E o `painel` a gente **lê** que existe. Não abre."*

Neste notebook você vai:

1. **Consultar o crt.sh** com `urllib` (plano B: o JSON da Isca embutido).
2. **Fazer DNS sem `dig`**: primeiro com `socket`, depois montando o pacote à mão com `struct`.
3. **Ler SPF e DMARC** e dar uma nota de "facilidade de falsificação".
4. **Montar a tabela de arestas e o grafo** com hubs e nós históricos.
5. **Jogar o quiz "passivo ou ativo?"**.

🐍 Só biblioteca padrão. Tudo do caso é fictício.

## 1. Certificate Transparency: o crt.sh

O Passo 3 do original usa `curl ... | jq`. Em Python é o mesmo: baixar o JSON, pegar `name_value`, quebrar por linha, tirar o `*.`, deduplicar.

Plano A: um domínio **real** e público, para você ver a resposta de verdade. Plano B (sempre, para o caso): o JSON da Isca, no formato real do crt.sh.

In [ ]:
import json, urllib.request, urllib.parse

CRT_ISCA = json.loads(r"""[{"id": 90000000001, "issuer_name": "C=XX, O=CA Gratuita Exemplo, CN=E1", "common_name": "serraazul-pix.example", "name_value": "serraazul-pix.example\nwww.serraazul-pix.example\nserraazul-seguranca.example\npainel.serraazul-pix.example", "entry_timestamp": "2026-10-04T19:12:41.120", "not_before": "2026-10-04T18:12:41", "not_after": "2027-01-02T18:12:40", "serial_number": "0aa1fict0001"}, {"id": 90000000002, "issuer_name": "C=XX, O=CA Gratuita Exemplo, CN=E1", "common_name": "mail.serraazul-seguranca.example", "name_value": "mail.serraazul-seguranca.example", "entry_timestamp": "2026-10-08T23:58:03.881", "not_before": "2026-10-08T22:58:03", "not_after": "2027-01-06T22:58:02", "serial_number": "0aa1fict0002"}, {"id": 80000000010, "issuer_name": "C=XX, O=CA Comercial Exemplo, CN=OV R3", "common_name": "serraazul.example", "name_value": "serraazul.example\nwww.serraazul.example", "entry_timestamp": "2025-11-20T10:30:00.000", "not_before": "2025-11-20T00:00:00", "not_after": "2026-11-20T23:59:59", "serial_number": "07leg0001"}, {"id": 80000000011, "issuer_name": "C=XX, O=CA Comercial Exemplo, CN=OV R3", "common_name": "internet.serraazul.example", "name_value": "internet.serraazul.example\napp.serraazul.example", "entry_timestamp": "2025-11-20T10:31:00.000", "not_before": "2025-11-20T00:00:00", "not_after": "2026-11-20T23:59:59", "serial_number": "07leg0002"}, {"id": 70000000005, "issuer_name": "C=XX, O=CA Comercial Exemplo, CN=OV R2", "common_name": "intranet-antiga.serraazul.example", "name_value": "intranet-antiga.serraazul.example", "entry_timestamp": "2021-03-01T09:00:00.000", "not_before": "2021-03-01T00:00:00", "not_after": "2022-03-01T23:59:59", "serial_number": "05leg0099"}]""")


def crtsh(dominio: str, timeout=30):
    """Consulta o crt.sh (curinga %.dominio). Devolve lista de registros ou None se falhar."""
    url = "https://crt.sh/?" + urllib.parse.urlencode({"q": "%." + dominio, "output": "json"})
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "estudo-investigador-cyber/1.0"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.load(r)
    except Exception as e:                       # crt.sh dá timeout com frequência: é normal
        print(f"   crt.sh indisponível agora ({type(e).__name__}). Tente mais tarde ou use o formulário web.")
        return None


def hosts_do_ct(registros):
    """= jq -r '.[].name_value' | tr A-Z a-z | sed 's/^\\*\\.//' | sort -u"""
    nomes = set()
    for r in registros:
        for n in r["name_value"].split("\n"):   # vários nomes por certificado
            nomes.add(n.lower().removeprefix("*."))
    return sorted(nomes)


#@markdown Domínio real para o plano A (público, institucional):
DOMINIO_REAL = "registro.br"  #@param {type:"string"}
real = crtsh(DOMINIO_REAL, timeout=20)
if real:
    hs = hosts_do_ct(real)
    print(f"{DOMINIO_REAL}: {len(real)} certificados, {len(hs)} nomes únicos. Primeiros:", hs[:8])

print("\n=== Operação Isca (fictício) ===")
for h in hosts_do_ct(CRT_ISCA):
    print("  ", h)

In [ ]:
# Quem divide certificado com quem? Cada certificado com mais de um "domínio registrado" vira aresta.
def registravel(host):
    """Simplificação: os dois últimos rótulos (serve para .example; para .com.br use três)."""
    return ".".join(host.split(".")[-2:])

print(f"{'emitido (not_before)':22s} {'emissor':34s} nomes")
for r in sorted(CRT_ISCA, key=lambda x: x["not_before"]):
    nomes = r["name_value"].split("\n")
    dominios = sorted({registravel(n) for n in nomes})
    marca = "  🔗 COMPARTILHADO" if len(dominios) > 1 else ""
    print(f"{r['not_before']:22s} {r['issuer_name'][-30:]:34s} {', '.join(nomes)}{marca}")

> 🔍 O certificado `id 90000000001` (emitido em 4/out, 18h12 UTC = 15h12 BRT, na véspera dos SMS) cobre `serraazul-pix.example` **e** `serraazul-seguranca.example`. É uma aresta forte: quem pediu o certificado controlava os dois nomes. O `painel.serraazul-pix.example` entra no mapa como **observado no CT, não visitado**.

## 2. DNS sem `dig`

### 2a. O jeito fácil: `socket`
`socket.getaddrinfo` pergunta ao *resolver* do sistema. Só devolve endereços (A/AAAA), mas já serve para o Passo 4 (host → IP).

In [ ]:
import socket

def resolver_a(host):
    """Devolve a lista de IPv4 de um host, ou [] se não resolver (host histórico?)."""
    try:
        return sorted({info[4][0] for info in socket.getaddrinfo(host, None, socket.AF_INET)})
    except OSError:
        return []

for h in ["registro.br", "serraazul-pix.example"]:
    ips = resolver_a(h)
    print(f"{h:25s} -> {ips if ips else 'não resolve (sem rede, ou .example não existe na internet)'}")

### 2b. O jeito de quem quer entender: o pacote DNS à mão

Uma consulta DNS é um pacote UDP: **cabeçalho** de 12 bytes + a **pergunta** (o nome em pedaços, o tipo e a classe). A resposta repete a pergunta e traz os registros. Nomes na resposta podem vir **comprimidos**: um ponteiro `0xC0xx` diz "o resto do nome está no byte xx".

In [ ]:
import struct, random

TIPOS = {1: "A", 2: "NS", 5: "CNAME", 6: "SOA", 15: "MX", 16: "TXT", 28: "AAAA"}


def montar_consulta(nome: str, tipo: int) -> bytes:
    ident = random.randint(0, 65535)                         # id para casar pergunta e resposta
    cab = struct.pack(">HHHHHH", ident, 0x0100, 1, 0, 0, 0)  # flags 0x0100 = "recursão desejada"
    qname = b"".join(bytes([len(p)]) + p.encode() for p in nome.strip(".").split(".")) + b"\x00"
    return cab + qname + struct.pack(">HH", tipo, 1)         # classe 1 = IN (internet)


def ler_nome(msg: bytes, i: int):
    """Lê um nome a partir da posição i, seguindo ponteiros de compressão. Devolve (nome, nova_posição)."""
    partes, saltou, fim = [], False, i
    while True:
        tam = msg[i]
        if tam == 0:                                          # fim do nome
            i += 1
            break
        if tam & 0xC0 == 0xC0:                                # ponteiro: 2 bytes, 14 bits de offset
            if not saltou:
                fim = i + 2
            i = ((tam & 0x3F) << 8) | msg[i + 1]
            saltou = True
            continue
        partes.append(msg[i + 1:i + 1 + tam].decode("ascii", "replace"))
        i += 1 + tam
    return ".".join(partes), (fim if saltou else i)


def ler_resposta(msg: bytes):
    """Decodifica as respostas: devolve lista de (nome, tipo, ttl, valor legível)."""
    _, flags, qd, an, _, _ = struct.unpack(">HHHHHH", msg[:12])
    i = 12
    for _ in range(qd):                                       # pula a(s) pergunta(s)
        _, i = ler_nome(msg, i)
        i += 4
    saida = []
    for _ in range(an):
        nome, i = ler_nome(msg, i)
        tipo, _, ttl, tam = struct.unpack(">HHIH", msg[i:i + 10])
        i += 10
        dado = msg[i:i + tam]
        if tipo == 1:
            valor = ".".join(map(str, dado))
        elif tipo == 16:                                      # TXT: uma ou mais strings com prefixo de tamanho
            j, txt = 0, []
            while j < len(dado):
                txt.append(dado[j + 1:j + 1 + dado[j]].decode("utf-8", "replace"))
                j += 1 + dado[j]
            valor = '"' + "".join(txt) + '"'
        elif tipo == 15:
            valor = f"{struct.unpack('>H', dado[:2])[0]} {ler_nome(msg, i + 2)[0]}"
        elif tipo in (2, 5):
            valor = ler_nome(msg, i)[0]
        else:
            valor = dado.hex()
        saida.append((nome, TIPOS.get(tipo, tipo), ttl, valor))
        i += tam
    return flags, saida


def dig(nome, tipo="A", servidor="1.1.1.1"):
    """Um 'dig nome tipo +noall +answer' de bolso. Devolve [] se não houver rede."""
    cod = {v: k for k, v in TIPOS.items()}[tipo]
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.settimeout(5)
        s.sendto(montar_consulta(nome, cod), (servidor, 53))
        flags, resp = ler_resposta(s.recv(4096))
        return resp
    except OSError as e:
        print(f"   (sem rede para DNS: {type(e).__name__})")
        return []


for tipo in ("A", "MX", "TXT"):
    for linha in dig("registro.br", tipo):
        print(*linha, sep="\t")
for linha in dig("_dmarc.registro.br", "TXT"):
    print(*linha, sep="\t")

In [ ]:
# OPCIONAL: os comandos do original (o teste pula linhas com !)
!apt-get -qq install -y dnsutils jq > /dev/null
!dig registro.br A +noall +answer
!dig _dmarc.registro.br TXT +short
!dig +trace registro.br | tail -15
print("(fora do Colab, as linhas com ! foram puladas)")

## 3. SPF e DMARC: quão fácil é falsificar o domínio?

As saídas abaixo são as do `dns-isca.txt` (fictícias, coerentes com os retratos do Dia 13). O código lê as tags do DMARC como a RFC 7489, seção 6.3, define.

In [ ]:
DNS_ISCA = r"""; Saídas ilustrativas (fictícias) de dig — Operação Isca, Dia 30 (consulta de 2026-10-09, antes da suspensão da hospedagem)
; dig serraazul.example A +noall +answer
serraazul.example.		300	IN	A	192.0.2.10
; dig serraazul.example MX +short
10 mx.serraazul.example.
; dig serraazul.example NS +short
ns1.provedor-dns.example.
; dig serraazul.example SOA +short
ns1.provedor-dns.example. ti.serraazul.example. 2025112001 7200 3600 1209600 300
; dig serraazul.example TXT +short
"v=spf1 ip4:192.0.2.0/24 -all"
"google-site-verification=FICTICIO123"
; dig _dmarc.serraazul.example TXT +short
"v=DMARC1; p=none; rua=mailto:dmarc@serraazul.example"
; dig intranet-antiga.serraazul.example A +short
192.0.2.44
; dig serraazul-pix.example A +short
203.0.113.77
; dig serraazul-pix.example NS +short
ns1.dnsbarato.example.
ns2.dnsbarato.example.
; dig serraazul-pix.example MX +short
; (sem resposta)
; dig serraazul-seguranca.example A +short
203.0.113.77
; dig serraazul-seguranca.example NS +short
ns1.dnsbarato.example.
; dig serraazul-seguranca.example MX +short
10 mail.serraazul-seguranca.example.
; dig coopserra-azul.example A +short
; (sem resposta: só NS de estacionamento)
; dig coopserra-azul.example NS +short
ns1.registrar-parking.example.
; dig -x 203.0.113.77 +short
vps-77.hospedagem-compartilhada.example.
; Team Cymru (formato real, dados fictícios; ASNs da faixa de documentação RFC 5398)
AS      | IP               | BGP Prefix          | CC | Registry | Allocated  | AS Name
64501   | 203.0.113.77     | 203.0.113.0/24      | XX | ficticio | 2019-01-01 | HOSPEDAGEM-COMPARTILHADA-EXEMPLO (fictício)
64500   | 192.0.2.10       | 192.0.2.0/24        | BR | ficticio | 2010-01-01 | CDN-EXEMPLO (fictício)
64502   | 192.0.2.44       | 192.0.2.0/24        | BR | ficticio | 2010-01-01 | COOPERATIVA-SERRA-AZUL (fictício)
"""

# Separa "; comando" e as linhas de resposta que vêm logo abaixo
respostas, atual = {}, None
for linha in DNS_ISCA.splitlines():
    if linha.startswith("; dig "):
        atual = linha[2:].strip()
        respostas[atual] = []
    elif atual and linha and not linha.startswith(";"):
        respostas[atual].append(linha.strip())

spf = next(v for v in respostas["dig serraazul.example TXT +short"] if "v=spf1" in v)
dmarc = respostas["dig _dmarc.serraazul.example TXT +short"][0]
print("SPF:  ", spf)
print("DMARC:", dmarc)


def tags_dmarc(txt):
    """'v=DMARC1; p=none; rua=...' -> dicionário de tags."""
    return dict(par.strip().split("=", 1) for par in txt.strip('"').split(";") if "=" in par)


t = tags_dmarc(dmarc)
politica = t.get("p", "ausente")
risco = {"reject": "baixo", "quarantine": "médio", "none": "ALTO"}.get(politica, "ALTO")
print(f"\nPolítica DMARC: p={politica} -> facilidade de falsificação do domínio: {risco}")
print("Relatórios agregados vão para:", t.get("rua", "(ninguém recebe)"))
print("SPF termina em", spf.strip('"').split()[-1], "-> '-all' manda rejeitar remetentes fora da lista,"
      " mas sem DMARC p=reject muitos provedores só marcam.")

> ✍️ **A frase do Passo 7.3, para a Marta:** *"O DMARC de serraazul.example está em `p=none`: e-mails falsificados com o remetente da cooperativa são só monitorados, não recusados, então um golpista pode usar o próprio domínio da Serra Azul no remetente. Recomendação: acompanhar os relatórios `rua` por algumas semanas e subir para `p=quarantine` e depois `p=reject`."*

## 4. A tabela de arestas e o grafo

Cada linha é uma aresta **com rótulo** e **com a fonte** que a produziu (o artefato do original: "uma linha por aresta, com o comando ou a fonte").

In [ ]:
import csv

ARESTAS = [
    # Source, Target, Label, fonte
    ("serraazul.example", "www.serraazul.example", "CT", "crt.sh id 80000000010"),
    ("serraazul.example", "internet.serraazul.example", "CT", "crt.sh id 80000000011"),
    ("serraazul.example", "intranet-antiga.serraazul.example", "CT (cert. vencido 2022)", "crt.sh id 70000000005"),
    ("intranet-antiga.serraazul.example", "192.0.2.44", "A (ainda ativo)", "dig intranet-antiga.serraazul.example A; exposições do Dia 23 NÃO abertas"),
    ("192.0.2.44", "AS64502", "ASN", "Team Cymru"),
    ("serraazul.example", "192.0.2.10", "A", "dig serraazul.example A"),
    ("serraazul.example", "mx.serraazul.example", "MX", "dig serraazul.example MX"),
    ("serraazul.example", "ns1.provedor-dns.example", "NS", "dig serraazul.example NS"),
    ("192.0.2.10", "AS64500", "ASN", "Team Cymru (CDN)"),
    ("serraazul-pix.example", "serraazul-seguranca.example", "SAN do mesmo certificado", "crt.sh id 90000000001"),
    ("serraazul-pix.example", "www.serraazul-pix.example", "CT", "crt.sh id 90000000001"),
    ("serraazul-pix.example", "painel.serraazul-pix.example", "CT (observado, não visitado)", "crt.sh id 90000000001"),
    ("serraazul-pix.example", "203.0.113.77", "A", "dig serraazul-pix.example A"),
    ("serraazul-pix.example", "ns1.dnsbarato.example", "NS", "dig serraazul-pix.example NS"),
    ("serraazul-seguranca.example", "203.0.113.77", "A", "dig serraazul-seguranca.example A"),
    ("serraazul-seguranca.example", "ns1.dnsbarato.example", "NS", "dig serraazul-seguranca.example NS"),
    ("serraazul-seguranca.example", "mail.serraazul-seguranca.example", "MX", "dig ... MX; crt.sh id 90000000002"),
    ("203.0.113.77", "AS64501", "ASN", "Team Cymru (hospedagem compartilhada)"),
    ("serraazul-pix.example", "coopserra-azul.example", "mesmo ID de analytics G-ISCA7K2Q9P", "Dias 19-26 (capturas)"),
    ("coopserra-azul.example", "ns1.registrar-parking.example", "NS", "dig coopserra-azul.example NS"),
]

with open("arestas-isca.csv", "w", newline="", encoding="utf-8") as f:   # pronto para o Gephi
    w = csv.writer(f)
    w.writerow(["Source", "Target", "Label", "Fonte"])
    w.writerows(ARESTAS)
print("💾 arestas-isca.csv:", len(ARESTAS), "arestas")

# Grau de cada nó: quem é hub?
from collections import Counter
grau = Counter()
for a, b, *_ in ARESTAS:
    grau[a] += 1
    grau[b] += 1
print("\nNós com mais conexões:")
for no, g in grau.most_common(5):
    print(f"  {g}  {no}")

In [ ]:
# Grafo em texto: cada domínio e o que pendura nele
from collections import defaultdict
filhos = defaultdict(list)
for a, b, rot, _ in ARESTAS:
    filhos[a].append((rot, b))

def desenhar(no, prefixo="", visto=None):
    visto = visto or set()
    for i, (rot, alvo) in enumerate(filhos.get(no, [])):
        ultimo = i == len(filhos[no]) - 1
        marca = "└─" if ultimo else "├─"
        tracejado = " ┄ histórico" if "histórico" in rot else ""
        print(f"{prefixo}{marca}[{rot}]→ {alvo}{tracejado}")
        if alvo not in visto:
            visto.add(alvo)
            desenhar(alvo, prefixo + ("   " if ultimo else "│  "), visto)

for raiz in ("serraazul.example", "serraazul-pix.example", "coopserra-azul.example"):
    print(raiz)
    desenhar(raiz)
    print()

print("Leitura: 203.0.113.77 / AS64501 parecem o 'hub' dos sites falsos. Mas é hospedagem compartilhada:")
print("mesmo IP e mesmo AS valem pouco sozinhos. A aresta forte entre pix e seguranca é o certificado;")
print("entre pix e coopserra-azul, o ID de analytics (achado nos Dias 19-26).")

## 5. 🎮 Quiz: o que o mapa sustenta?

Para cada afirmação, responda **S** (o mapa sustenta, com a confiança dita) ou **N** (não sustenta). `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
QUIZ = [
    ("serraazul-pix e serraazul-seguranca estão ligados por SAN do mesmo certificado.", "S",
     "Mesmo certificado = quem o pediu controlava os dois nomes. Aresta forte."),
    ("Os dois sites falsos estão no mesmo AS, logo são do mesmo operador.", "N",
     "Mesmo AS de hospedagem barata vale pouco sozinho: milhares de sites dividem o AS."),
    ("O histórico de 2019 do coopserra-azul.example mostra quem opera o kit hoje.", "N",
     "O ID de analytics liga o domínio ao caso atual; o titular de 2019 é de outra 'vida' do nome."),
    ("intranet-antiga.serraazul.example tem certificado vencido mas ainda resolve; vai no mapa com essa nota.", "S",
     "CT dá a história; o DNS diz que o host segue ativo. As exposições do Dia 23 continuam NÃO abertas."),
    ("painel.serraazul-pix.example existe, então posso abrir para confirmar.", "N",
     "Fica como 'observado no CT, não visitado'. Entrar seria acesso a sistema alheio."),
    ("O DMARC p=none da cooperativa facilita usar o remetente @serraazul.example num golpe.", "S",
     "p=none só monitora; não manda recusar e-mail falsificado."),
    ("O certificado foi emitido na véspera dos SMS: isso prova premeditação de uma pessoa específica.", "N",
     "A data é fato; 'quem' é atribuição de pessoa, fora do escopo."),
    ("Cada aresta do CSV tem uma fonte que permite reproduzi-la.", "S",
     "É o artefato do original: uma linha por aresta, com o comando ou a fonte."),
]

MODO_JOGO = True
acertos = 0
for i, (frase, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {frase}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   O mapa sustenta? (S/N): ").strip().upper()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"Gabarito: {certo}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)} — meta do Checkpoint BR: 6.")

## ✅ Antes de fechar o Dia 30

- [ ] Hosts do CT da Isca listados; aresta do certificado compartilhado anotada
- [ ] `painel` marcado como observado, não visitado; `intranet-antiga` com certificado vencido e host ativo
- [ ] Frase do DMARC escrita para a Marta
- [ ] `arestas-isca.csv` gerado (importe no Gephi se quiser o PNG)
- [ ] Quiz com pelo menos 6/8

**Amanhã (Dia 31):** os perfis falsos `@serraazul.oficial` e "Serra Azul Atendimento" estão atrás de login. Rafa vai perguntar se a Vigia pode criar um perfil de fachada, e a resposta começa por escrito.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>